# Build HTMLs

Executes `NYC.ipynb`, `Boston.ipynb`, `Chicago.ipynb`, and `DC.ipynb` end-to-end and writes a self-contained HTML next to each (`NYC.html`, `Boston.html`, `Chicago.html`, `DC.html`).

Each city notebook will download data, run clustering, and build the linked widgets — expect 1–5 minutes per notebook depending on dataset size and whether the trip data is already cached under `~/.cache/bike_network_traffic/`.

In [13]:
import shutil
import sys
import time
from pathlib import Path

from nbconvert import HTMLExporter
from nbconvert.preprocessors import ExecutePreprocessor
import nbformat

ROOT = Path.cwd()
NOTEBOOKS = ['NYC.ipynb', 'Boston.ipynb', 'Chicago.ipynb', 'DC.ipynb']
TIMEOUT_S = 1800
KERNEL_NAME = 'python3'

for nb in NOTEBOOKS:
    assert (ROOT / nb).is_file(), f'missing notebook: {nb}'
print('Will build:', NOTEBOOKS)
print('Working dir:', ROOT)

Will build: ['NYC.ipynb', 'Boston.ipynb', 'Chicago.ipynb', 'DC.ipynb']
Working dir: /Users/nicolasfernandez/Documents/bike_network_traffic


## Run + export

We execute each notebook **in place** with a fresh kernel via `ExecutePreprocessor` (so widget output is captured), then render with `HTMLExporter`. Setting `embed_images=True` makes the resulting HTML fully self-contained (no relative paths to data/, etc.).

In [ ]:
def build_html(nb_path: Path) -> Path:
    out_path = nb_path.with_suffix('.html')
    print(f'\n=== {nb_path.name} -> {out_path.name} ===', flush=True)
    t0 = time.time()

    nb = nbformat.read(nb_path, as_version=4)

    ep = ExecutePreprocessor(timeout=TIMEOUT_S, kernel_name=KERNEL_NAME)
    print(f'  executing... (timeout={TIMEOUT_S}s)', flush=True)
    ep.preprocess(nb, {'metadata': {'path': str(nb_path.parent)}})

    exporter = HTMLExporter()
    exporter.embed_images = True
    body, _ = exporter.from_notebook_node(nb)
    out_path.write_text(body, encoding='utf-8')

    dt = time.time() - t0
    size_mb = out_path.stat().st_size / 1e6
    print(f'  wrote {out_path.name} ({size_mb:.1f} MB) in {dt:.1f}s', flush=True)
    return out_path

results = {}
for name in NOTEBOOKS:
    try:
        results[name] = ('ok', build_html(ROOT / name))
    except Exception as e:
        print(f'  FAILED: {type(e).__name__}: {e}', flush=True)
        results[name] = ('error', e)


=== NYC.ipynb -> NYC.html ===
  executing... (timeout=1800s)
  wrote NYC.html (22.6 MB) in 28.4s

=== Boston.ipynb -> Boston.html ===
  executing... (timeout=1800s)
  wrote Boston.html (15.1 MB) in 15.3s

=== Chicago.ipynb -> Chicago.html ===
  executing... (timeout=1800s)
  wrote Chicago.html (16.9 MB) in 18.8s

=== DC.ipynb -> DC.html ===
  executing... (timeout=1800s)


## Summary

In [ ]:
for name, (status, info) in results.items():
    if status == 'ok':
        print(f'  OK   {name:<14} -> {info.name}')
    else:
        print(f'  FAIL {name:<14} -> {type(info).__name__}: {info}')